# TP3 — Agent مبارك ولد حميدة : Mémoire, Outils et Sources
**LangChain · LangSmith · APIs externes · Continuité du personnage Hassaniya**

Ce notebook est la suite directe du **TP2**. Il transforme le personnage مبارك ولد حميدة (تاجر الأتاي والتمر) en agent LangChain doté d'outils vivants, d'une mémoire épisodique et de sources traçables dans LangSmith.

## 0. Installation et configuration

In [ ]:
# Groq est GRATUIT — créer un compte sur https://console.groq.com → API Keys
!pip install -q langchain langgraph langchain-community langchain-groq langsmith requests datasets huggingface_hub

In [1]:
import os

# === Clés API — utiliser Colab Secrets (icône clé dans le panneau gauche) ===
# GROQ est GRATUIT : https://console.groq.com → créer un compte → API Keys
# Ne jamais mettre les clés en clair dans le notebook rendu.
try:
    from google.colab import userdata
    os.environ["GROQ_API_KEY"]      = userdata.get("GROQ_API_KEY")
    os.environ["LANGSMITH_API_KEY"] = userdata.get("LANGSMITH_API_KEY")
    print("✅ Clés chargées depuis Colab Secrets")
except Exception:
    # Exécution locale — définir avant de lancer :
    #   set GROQ_API_KEY=gsk_...   (Windows)
    #   export GROQ_API_KEY=gsk_... (Linux/Mac)
    print("⚠️  Colab Secrets non disponibles — définissez GROQ_API_KEY et LANGSMITH_API_KEY")

# === LangSmith (optionnel mais recommandé pour les captures) ===
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_PROJECT"] = "tp3-mbark-hassaniya-agent"

print("LangSmith project :", os.environ["LANGSMITH_PROJECT"])
print("GROQ_API_KEY définie :", bool(os.environ.get("GROQ_API_KEY")))

⚠️  Colab Secrets non disponibles — définissez GROQ_API_KEY et LANGSMITH_API_KEY
LangSmith project : tp3-mbark-hassaniya-agent
GROQ_API_KEY définie : False


---
## Exercice 1 : Carte du personnage

### 1.1 Reprise du personnage TP2

| Dimension | Détail |
|-----------|--------|
| **Nom** | مبارك ولد حميدة (Mbark ould Hmeyda) |
| **Langue** | Hassaniya (dialecte arabe mauritanien) + expressions arabes classiques + quelques mots français |
| **Métier** | Vieux marchand de thé (أتاي), dattes (التمر) et warka (وركة) au marché central de Nouakchott |
| **Univers** | Marché, thé, dattes, famille, prix, Nouakchott, prières, météo, proverbes |
| **Registre** | Chaleureux, sage, paternel, religieux — formules : "هيه", "إن شاء الله", "الله يبارك فيك" |
| **Limites** | Ne jamais inventer un prix, un lieu ou une information factuelle. Si incertain : "والله ما نعرف " |

### 1.2 Profil système

In [1]:
CHARACTER_PROFILE = """
Tu es مبارك ولد حميدة (Mbark ould Hmeyda), vieux marchand de thé (أتاي),
de dattes (التمر) et de warka (وركة) au marché central de Nouakchott, Mauritanie.

Langue : Hassaniya (dialecte arabe mauritanien), avec quelques mots arabes classiques
         et rarement quelques mots français.

Ton : chaleureux, sage, paternel, ancré dans la foi islamique.

Expressions typiques :
  - Ouverture   : "هيه", "السلام عليكم", "اشحالك", "شماسي"
  - Fermeture   : "الله يبارك فيك", "إن شاء الله", "ربي يحفظك", "الله يعين"
  - Confirmation: "وف ذاك", "صحيح", "ايو"
  - Refus poli  : "ذاك شوي", "مانكد", "ابدي"

Règles :
  1. Réponds TOUJOURS en Hassaniya en priorité.
  2. Ajoute une courte note en français entre parenthèses si la réponse contient
     une information factuelle (météo, prix, prières).
  3. IMPORTANT : si tu as besoin d'un outil, appelle-le DIRECTEMENT sans aucun texte
     avant l'appel. Présente le résultat seulement APRÈS avoir reçu la réponse de l'outil.
  4. Si tu ne sais pas : dis "والله ما نعرف ".
  5. Ne jamais inventer un prix, des coordonnées ou un horaire.
  6. Cite souvent un proverbe hassaniya pour illustrer un conseil.

Format de réponse attendu (après l'appel d'outil) :
  [Réponse en Hassaniya avec les données de l'outil]
  (Note en français si nécessaire)
  Source : [nom de la source]
  Confiance : forte / moyenne / faible
"""

print("Profil du personnage défini ✅")

Profil du personnage défini ✅


---
## Exercice 2 : Character With Live Tools

Cinq outils sont créés, dont **trois APIs externes** (Open-Meteo, AlAdhan, Frankfurter) et deux outils locaux (proverbes, dataset TP2).

### Justification des APIs

| API | Justification (lien personnage) |
|-----|---------------------------------|
| **Open-Meteo** (météo) | Un marchand de marché en plein air surveille la chaleur, le vent de sable (الغبر) et la pluie pour décider d'ouvrir ou rester chez lui |
| **AlAdhan** (prières) | مبارك organise sa journée entière autour des 5 prières — c'est une information centrale dans son univers |
| **Frankfurter** (change) | Il traite avec des touristes et commerçants étrangers ; convertir euros/dollars en MRU (ouguiya) est utile chaque jour |

**Distinction RAG vs API :**
- L'API fournit des données **en temps réel** (météo du jour, horaire de prière du jour, taux du jour).
- Le dataset TP2 / les proverbes fournissent des connaissances **culturelles stables** qui ne changent pas.

### Import des bibliothèques

In [2]:
import json
import requests
from langchain_core.tools import tool

print("Bibliothèques importées ✅")

ModuleNotFoundError: No module named 'langchain_core'

### Outil 1 — Météo (Open-Meteo API)

In [ ]:
@tool
def get_weather(city: str = "Nouakchott") -> dict:
    """Obtenir la météo actuelle d'une ville : température, vent, humidité.
    Utile pour conseiller sur les conditions de marché ou de voyage en Mauritanie."""
    # Géocodage via Open-Meteo Geocoding
    geo = requests.get(
        "https://geocoding-api.open-meteo.com/v1/search",
        params={"name": city, "count": 1, "language": "fr"},
        timeout=10
    ).json()
    if not geo.get("results"):
        return {"erreur": f"Ville '{city}' introuvable"}
    loc = geo["results"][0]
    # Météo actuelle
    weather = requests.get(
        "https://api.open-meteo.com/v1/forecast",
        params={
            "latitude": loc["latitude"],
            "longitude": loc["longitude"],
            "current": "temperature_2m,relative_humidity_2m,wind_speed_10m,precipitation",
            "timezone": "auto"
        },
        timeout=10
    ).json()
    c = weather["current"]
    return {
        "ville": city,
        "temperature_c": c["temperature_2m"],
        "humidite_pct": c["relative_humidity_2m"],
        "vent_kmh": c["wind_speed_10m"],
        "precipitations_mm": c["precipitation"],
        "source": "Open-Meteo API (open-meteo.com)"
    }

# Test
print("Test Outil 1 — Météo Nouakchott :")
print(get_weather.invoke({"city": "Nouakchott"}))

### Outil 2 — Horaires de prière (AlAdhan API)

In [ ]:
@tool
def get_prayer_times(city: str = "Nouakchott", country: str = "Mauritania") -> dict:
    """Retourner les horaires des 5 prières quotidiennes selon la ville et le pays.
    Essentiel pour un marchand musulman qui structure sa journée autour des prières."""
    r = requests.get(
        "https://api.aladhan.com/v1/timingsByCity",
        params={"city": city, "country": country, "method": 3},
        timeout=10
    )
    r.raise_for_status()
    t = r.json()["data"]["timings"]
    return {
        "ville": city,
        "Fajr (الفجر)": t["Fajr"],
        "Dhuhr (الظهر)": t["Dhuhr"],
        "Asr (العصر)": t["Asr"],
        "Maghrib (المغرب)": t["Maghrib"],
        "Isha (العشاء)": t["Isha"],
        "source": "AlAdhan Prayer Times API (aladhan.com)"
    }

print("Test Outil 2 — Prières Nouakchott :")
print(get_prayer_times.invoke({"city": "Nouakchott", "country": "Mauritania"}))

### Outil 3 — Taux de change (Frankfurter API)

In [ ]:
@tool
def get_exchange_rate(from_currency: str = "EUR", to_currency: str = "USD", amount: float = 1.0) -> dict:
    """Convertir des devises : EUR, USD, MAD vers d'autres monnaies.
    Utile pour un marchand mauritanien qui compare les prix avec des clients étrangers.
    Note: MRU (ouguiya mauritanien) n'est pas toujours dans l'API; utiliser USD comme référence intermédiaire."""
    r = requests.get(
        "https://api.frankfurter.app/latest",
        params={"from": from_currency, "to": to_currency, "amount": amount},
        timeout=10
    )
    r.raise_for_status()
    data = r.json()
    converted = data["rates"].get(to_currency)
    if converted is None:
        return {"erreur": f"Devise '{to_currency}' non supportée. Devises disponibles: EUR, USD, GBP, MAD, SAR..."}
    return {
        "de": f"{amount} {from_currency}",
        "vers": f"{converted:.4f} {to_currency}",
        "taux_unitaire": round(converted / amount, 4),
        "date": data["date"],
        "source": "Frankfurter Exchange Rate API (frankfurter.app)"
    }

print("Test Outil 3 — Taux de change EUR → USD :")
print(get_exchange_rate.invoke({"from_currency": "EUR", "to_currency": "USD", "amount": 100}))

### Outil 4 — Proverbes Hassaniya (dataset local + HuggingFace)

In [3]:
# Base de proverbes hassaniya intégrée
PROVERBES = [
    {"hassaniya": "الصبر مفتاح الخير",
     "fr": "La patience est la clé du soulagement",
     "themes": ["patience", "conseil", "difficulté"]},
    {"hassaniya": "الزبون هو البطرون",
     "fr": "Le client est ton maître",
     "themes": ["commerce", "client", "marché", "vente"]},
    {"hassaniya": "البركة في البكور",
     "fr": "La bénédiction est dans le lever tôt",
     "themes": ["travail", "matin", "marché", "bénédiction"]},
    {"hassaniya": "اللي يعدل أتاي بشروطو يجبر طعمتو",
     "fr": "Celui qui fait le thé avec patience en comprend le goût",
     "themes": ["thé", "patience", "atay", "culture"]},
    {"hassaniya": "الحق يغلب",
     "fr": "La vérité triomphe toujours",
     "themes": ["vérité", "honnêteté", "commerce", "droit"]},
    {"hassaniya": "العلم نور",
     "fr": "Le savoir est une lumière",
     "themes": ["savoir", "éducation", "apprentissage"]},
    {"hassaniya": "من يزرع خير يحصد خير",
     "fr": "Qui sème le bien récolte le bien",
     "themes": ["karma", "bonté", "récompense", "foi"]},
    {"hassaniya": "الوقت من ذهب",
     "fr": "Le temps c'est de l'or",
     "themes": ["temps", "travail", "marché", "commerce"]},
    {"hassaniya": "النية تحكم",
     "fr": "L'intention décide de tout",
     "themes": ["intention", "foi", "islam", "morale"]},
    {"hassaniya": "كل حد يعرف حقيقتو",
     "fr": "Chacun connaît ce qu'il porte en lui",
     "themes": ["connaissance", "soi", "introspection"]},
    {"hassaniya": "اللي عنده أتاي عنده ضيف",
     "fr": "Celui qui a du thé a toujours un hôte",
     "themes": ["hospitalité", "thé", "atay", "famille"]},
    {"hassaniya": "الصدق طريق السلامة",
     "fr": "L'honnêteté est le chemin de la paix",
     "themes": ["honnêteté", "paix", "commerce", "morale"]}
]

@tool
def search_proverbs(query: str) -> list:
    """Rechercher des proverbes hassaniya liés à un thème : patience, commerce, thé, famille, sagesse, temps.
    Retourne les 3 proverbes les plus pertinents avec traduction et contexte d'usage."""
    query_lower = query.lower()
    scored = []
    for p in PROVERBES:
        score = 0
        # Score par thèmes
        score += sum(2 for t in p["themes"] if t in query_lower)
        # Score par mots dans le texte
        score += sum(1 for w in query_lower.split() if w in p["hassaniya"] + p["fr"].lower())
        if score > 0:
            scored.append((score, p))
    scored.sort(key=lambda x: x[0], reverse=True)
    results = [p for _, p in scored[:3]]
    if not results:
        import random
        results = random.sample(PROVERBES, min(2, len(PROVERBES)))
    return results

print("Test Outil 4 — Proverbes (thème : patience) :")
for p in search_proverbs.invoke({"query": "patience"}): print(" •", p["hassaniya"], "→", p["fr"])

NameError: name 'tool' is not defined

### Outil 5 — Géocodage / Carte (OpenStreetMap Nominatim)

In [ ]:
@tool
def geocode_place(place_name: str) -> dict:
    """Trouver les coordonnées et informations d'un lieu en Mauritanie ou ailleurs.
    Utile pour expliquer où se trouve un marché, un quartier ou une ville."""
    headers = {"User-Agent": "tp3-mbark-hassaniya-agent/1.0 (educational project)"}
    # Essai avec Mauritanie d'abord
    for query in [f"{place_name} Mauritania", place_name]:
        r = requests.get(
            "https://nominatim.openstreetmap.org/search",
            params={"q": query, "format": "json", "limit": 1, "accept-language": "fr"},
            headers=headers, timeout=10
        )
        data = r.json()
        if data:
            loc = data[0]
            return {
                "lieu": place_name,
                "nom_complet": loc.get("display_name", ""),
                "latitude": loc.get("lat"),
                "longitude": loc.get("lon"),
                "type": loc.get("type", ""),
                "source": "OpenStreetMap / Nominatim"
            }
    return {"erreur": f"Lieu '{place_name}' introuvable"}

print("Test Outil 5 — Géocodage (Nouakchott) :")
print(geocode_place.invoke({"place_name": "Marché Capital Nouakchott"}))

### Outil 6 — Modèle TP2 (Générateur Hassaniya authentique)

Le modèle fine-tuné du TP2 est chargé ici et encapsulé comme **outil LangChain**.  
Groq l'appelle quand il veut générer une réponse dans le style Hassaniya authentique de مبارك.

> **Rôle dans l'architecture :**  
> Groq = cerveau raisonneur → TP2 model = voix Hassaniya authentique

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

# ── Chargement du modèle TP2 ─────────────────────────────────────────────────
_tp2_model     = None
_tp2_tokenizer = None

def load_tp2_model():
    global _tp2_model, _tp2_tokenizer
    if _tp2_model is not None:
        return True  # déjà chargé

    # Priorité 1 : modèle local (si exécution locale ou uploadé dans Colab)
    import os
    local_path = "./mbark_model"
    if os.path.exists(local_path):
        print(f"Chargement depuis {local_path} ...")
        _tp2_tokenizer = AutoTokenizer.from_pretrained(local_path)
        _tp2_model     = AutoModelForCausalLM.from_pretrained(local_path)
        _tp2_model.eval()
        print("✅ Modèle TP2 chargé (local)")
        return True

    # Priorité 2 : HuggingFace Hub (si vous avez poussé le modèle — Ex7)
    hf_model_id = "Mokhtar-46/mbark-hassaniya-tp2-model"  # ← changer si nécessaire
    try:
        print(f"Chargement depuis HuggingFace Hub ({hf_model_id}) ...")
        _tp2_tokenizer = AutoTokenizer.from_pretrained(hf_model_id)
        _tp2_model     = AutoModelForCausalLM.from_pretrained(hf_model_id)
        _tp2_model.eval()
        print("✅ Modèle TP2 chargé (HuggingFace Hub)")
        return True
    except Exception as e:
        print(f"HuggingFace Hub non disponible : {e}")

    # Priorité 3 : W&B artifact (comme dans app.py du TP2)
    try:
        import wandb
        print("Chargement depuis W&B ...")
        run = wandb.init(project="hassaniya-mbark", job_type="inference")
        artifact = run.use_artifact(
            "oussallay200-esp/hassaniya-mbark/mbark-hassaniya-model:latest"
        )
        artifact_dir = artifact.download()
        _tp2_tokenizer = AutoTokenizer.from_pretrained(artifact_dir)
        _tp2_model     = AutoModelForCausalLM.from_pretrained(artifact_dir)
        _tp2_model.eval()
        wandb.finish()
        print("✅ Modèle TP2 chargé (W&B)")
        return True
    except Exception as e:
        print(f"W&B non disponible : {e}")

    print("⚠️  Modèle TP2 non chargé — l'outil génération Hassaniya sera désactivé")
    return False

# Tenter le chargement
tp2_available = load_tp2_model()

# ── Fonction de génération ────────────────────────────────────────────────────
def _generate_hassaniya(prompt: str, max_new_tokens: int = 80) -> str:
    """Utilise le modèle TP2 pour générer du texte Hassaniya."""
    if not tp2_available or _tp2_model is None:
        return "والله ما نعرف "  
    
    full_prompt = f"سؤال: {prompt}\nجواب: "
    inputs = _tp2_tokenizer(full_prompt, return_tensors="pt")

    with torch.no_grad():
        output = _tp2_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.8,
            repetition_penalty=1.3,
            do_sample=True,
            pad_token_id=_tp2_tokenizer.eos_token_id
        )
    decoded = _tp2_tokenizer.decode(output[0], skip_special_tokens=True)
    # Extraire uniquement la partie après "جواب: "
    answer = decoded.split("جواب: ")[-1].strip()
    return answer

# ── Outil LangChain ───────────────────────────────────────────────────────────
@tool
def generate_hassaniya_response(topic: str) -> str:
    """Générer une réponse en Hassaniya authentique avec le modèle fine-tuné TP2.
    Utiliser pour les réponses culturelles, conseils, sagesse, ou expressions typiques
    de مبارك quand aucune API externe n'est nécessaire."""
    return _generate_hassaniya(topic)

# Test si modèle disponible
if tp2_available:
    test = generate_hassaniya_response.invoke({"topic": "اشحالك اليوم"})
    print("Test Outil 6 — Génération TP2 :")
    print(f"  → {test}")
else:
    print("Outil 6 défini (modèle non chargé — sera ignoré par l'agent)")

---
## Exercice 3 : Character Field Agent

Création de l'agent avec `create_react_agent` de LangGraph. L'agent reçoit :
- le profil du personnage (system prompt),
- les 5 outils,
- un `MemorySaver` comme checkpointer (mémoire par `thread_id`).

In [ ]:
from langchain_groq import ChatGroq
from langgraph.prebuilt import create_react_agent
from langgraph.checkpoint.memory import MemorySaver
from langchain_core.messages import HumanMessage

# Limites Groq free tier (tokens/jour) :
#   llama-3.3-70b-versatile  →  100 000  (limite atteinte rapidement)
#   llama-3.1-8b-instant     →  500 000  ← 5x plus, recommandé pour les tests
llm = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0.7,
    model_kwargs={"parallel_tool_calls": False}
)

# Liste des outils — inclut le modèle TP2 si disponible
tools = [get_weather, get_prayer_times, get_exchange_rate,
         search_proverbs, geocode_place]

if tp2_available:
    tools.append(generate_hassaniya_response)
    print("✅ Outil TP2 ajouté")

checkpointer = MemorySaver()

agent = create_react_agent(
    model=llm,
    tools=tools,
    checkpointer=checkpointer,
    prompt=CHARACTER_PROFILE
)

print(f"Agent مبارك créé ✅  (llama-3.1-8b-instant · {len(tools)} outils)")
print("Outils :", [t.name for t in tools])

In [ ]:
def chat_with_mbark(user_input: str, thread_id: str = "default") -> dict:
    """Envoyer un message à مبارك et retourner la réponse structurée."""
    config = {"configurable": {"thread_id": thread_id}}
    result = agent.invoke(
        {"messages": [HumanMessage(content=user_input)]},
        config=config
    )
    messages = result["messages"]
    final_response = messages[-1].content

    # Extraire les appels d'outils
    tools_called = []
    sources_used = []
    for msg in messages:
        if hasattr(msg, "tool_calls") and msg.tool_calls:
            for tc in msg.tool_calls:
                tools_called.append(tc["name"])
        if hasattr(msg, "name") and msg.name in [t.name for t in tools]:
            try:
                content = json.loads(msg.content) if isinstance(msg.content, str) else msg.content
                if isinstance(content, dict) and "source" in content:
                    sources_used.append(content["source"])
                else:
                    sources_used.append(msg.name)
            except Exception:
                sources_used.append(msg.name)

    confidence = "forte" if tools_called else "moyenne"

    return {
        "response": final_response,
        "action": tools_called[0].upper() if tools_called else "DIRECT_RESPONSE",
        "tools_used": tools_called,
        "sources": sources_used if sources_used else ["Connaissance du personnage (TP2)"],
        "confidence": confidence,
        "thread_id": thread_id
    }

print("Fonction chat_with_mbark définie ✅")

In [ ]:
# === MISSIONS DE TERRAIN ===
missions = [
    ("شماسي الجو  في نواكشوط اليوم؟",         "mission-meteo",   "Météo"),
    (" صلاة المغرب الليلة اينت ؟",              "mission-salat",   "Prières"),
    ("كم 50 يورو بالدولار؟",           "mission-change",  "Change"),
    ("عندك مثل حساني عن الصبر في التجارة؟", "mission-proverb", "Proverbe"),
    (" سوق الكبير في نواكشوط منين ؟",           "mission-lieu",    "Lieu"),
]

for question, tid, label in missions:
    print(f"\n{'='*60}")
    print(f"Mission : {label}")
    print(f"سؤال : {question}")
    r = chat_with_mbark(question, thread_id=tid)
    print(f"مبارك : {r['response'][:350]}")
    print(f"  ↳ Action     : {r['action']}")
    print(f"  ↳ Outils     : {r['tools_used'] or 'Réponse directe'}")
    print(f"  ↳ Sources    : {r['sources']}")
    print(f"  ↳ Confiance  : {r['confidence']}")

---
## Exercice 4 : Character With Episodic Memory

La mémoire épisodique est gérée par le `MemorySaver` via `thread_id`. Deux conversations avec le même `thread_id` partagent l'historique ; deux `thread_id` différents sont totalement isolés.

### Test de mémoire — même thread_id

In [ ]:
THREAD_A = "visiteur-paris-001"

print("=== Conversation A — Tour 1 ===")
r1 = chat_with_mbark(
    "سلام مبارك، أنا جاي من باريس وما نحمل الحمان",
    thread_id=THREAD_A
)
print(f"مبارك : {r1['response']}\n")

print("=== Conversation A — Tour 2 (même thread_id) ===")
r2 = chat_with_mbark(
    " شتنصحني للسوق؟",
    thread_id=THREAD_A
)
print(f"مبارك : {r2['response']}")
print("\n→ Vérifier que مبارك mentionne 'باريس' ou 'الحمان' dans sa réponse au tour 2")

In [ ]:
THREAD_B = "inconnu-002"  # thread_id différent → pas de mémoire du visiteur parisien

print("=== Conversation B — thread_id différent ===")
r3 = chat_with_mbark(
    " شتنصحني للسوق؟",
    thread_id=THREAD_B
)
print(f"مبارك : {r3['response']}")
print("\n→ مبارك ne sait PAS que ce visiteur vient de Paris ni qu'il n'aime pas la chaleur")
print("→ Preuve que le checkpointer sépare bien les conversations par thread_id")

In [ ]:
# Vérification du checkpointer
state_a = agent.get_state({"configurable": {"thread_id": THREAD_A}})
state_b = agent.get_state({"configurable": {"thread_id": THREAD_B}})

print(f"Thread A ({THREAD_A}) — nombre de messages : {len(state_a.values.get('messages', []))}")
print(f"Thread B ({THREAD_B}) — nombre de messages : {len(state_b.values.get('messages', []))}")
print("\nMessages Thread A :")
for m in state_a.values.get("messages", []):
    role = getattr(m, "type", type(m).__name__)
    print(f"  [{role}] {str(m.content)[:100]}")

---
## Exercice 5 : Sources et datasets Hugging Face

Deux sources textuelles complètent les APIs :
1. **Dataset TP2** (`data/train.json`) — 9 500 paires Q/R en Hassaniya
2. **Proverbes intégrés** — 12 proverbes hassaniya (+ support optionnel du dataset HuggingFace `ahmed02mk/amthal-hassaniya`)

In [ ]:
# Chargement du dataset TP2 local
import os

tp2_data = []
data_path = "data/train.json"
if os.path.exists(data_path):
    with open(data_path, "r", encoding="utf-8") as f:
        tp2_data = json.load(f)
    print(f"Dataset TP2 chargé : {len(tp2_data)} exemples")
    print("Exemple :")
    print(json.dumps(tp2_data[0], ensure_ascii=False, indent=2))
else:
    print("⚠️  data/train.json non trouvé — le dataset local ne sera pas utilisé")

# Tentative de chargement du dataset HuggingFace
hf_proverbs = []
try:
    from datasets import load_dataset
    ds = load_dataset("ahmed02mk/amthal-hassaniya", split="train", trust_remote_code=True)
    hf_proverbs = list(ds)
    print(f"\nDataset HuggingFace chargé : {len(hf_proverbs)} proverbes")
    print("Exemple :", hf_proverbs[0])
except Exception as e:
    print(f"\nDataset HuggingFace non disponible ({e}) — utilisation des proverbes intégrés")

In [ ]:
@tool
def search_local_knowledge(query: str) -> list:
    """Rechercher dans la base de connaissances locale du TP2 : expressions hassaniya,
    conseils de marché, réponses culturelles, dialogues typiques de Nouakchott.
    Utiliser pour des questions culturelles, linguistiques ou de la vie quotidienne."""
    if not tp2_data:
        return [{"message": "Dataset TP2 non disponible"}]
    query_words = set(query.lower().split())
    scored = []
    for item in tp2_data:
        q_text = item.get("question", item.get("input", "")).lower()
        a_text = item.get("answer",   item.get("output", "")).lower()
        score = sum(1 for w in query_words if w in q_text + a_text)
        if score > 0:
            scored.append((score, item))
    scored.sort(key=lambda x: x[0], reverse=True)
    results = [item for _, item in scored[:3]]
    if not results:
        return [{"message": "Aucun résultat pertinent dans le dataset TP2"}]
    return results

# Ajouter cet outil à la liste
tools.append(search_local_knowledge)

# Recréer l'agent avec le nouvel outil
agent = create_react_agent(
    model=llm,
    tools=tools,
    checkpointer=checkpointer,
    prompt=CHARACTER_PROFILE
)

print(f"Agent mis à jour — {len(tools)} outils : {[t.name for t in tools]}")

# Test
print("\nTest RAG — 'thé marché' :")
results = search_local_knowledge.invoke({"query": "thé marché"})
for r in results[:2]:
    print(" •", json.dumps(r, ensure_ascii=False)[:120])

In [ ]:
def format_source_answer(response: str, action: str, sources: list,
                          confidence: str, langsmith_trace: str = "") -> dict:
    """Formater la réponse finale avec toutes les métadonnées de traçabilité."""
    return {
        "response":        response,
        "action":          action,
        "sources":         sources,
        "confidence":      confidence,
        "langsmith_trace": langsmith_trace
    }

# Exemple
example = format_source_answer(
    response="هيه، الطقس في نواكشوط اليوم 38 درجة لرياح متين، الله يعين!",
    action="get_weather",
    sources=["Open-Meteo API"],
    confidence="forte",
    langsmith_trace="https://smith.langchain.com/projects/tp3-mbark-hassaniya-agent"
)
print("Exemple de réponse formatée :")
print(json.dumps(example, ensure_ascii=False, indent=2))

---
## Exercice 6 : LangSmith — Observation des chaînes

LangSmith est activé depuis la cellule 0. Les captures suivantes doivent être ajoutées dans le rapport :

1. **Capture 1** — chaîne simple (réponse directe sans outil)
2. **Capture 2** — appel d'outil externe (ex: `get_weather`)
3. **Capture 3** — conversation avec mémoire épisodique (même `thread_id`)
4. **Capture 4** — réponse finale avec sources et confiance

Pour accéder à LangSmith : [smith.langchain.com](https://smith.langchain.com) → Projet `tp3-mbark-hassaniya-agent`

In [ ]:
print("=== Test LangSmith — Chaîne simple (sans outil) ===")
r_simple = chat_with_mbark("السلام عليكم مبارك، اشحالك؟", thread_id="langsmith-simple")
print(f"مبارك : {r_simple['response']}")
print(f"Outils : {r_simple['tools_used'] or 'Aucun (réponse directe)'}")
print("→ Capture 1 : chaîne simple à prendre dans LangSmith")

print("\n=== Test LangSmith — Appel d'outil externe ===")
r_tool = chat_with_mbark("شماسي الطقس في روصو اليوم؟", thread_id="langsmith-tool")
print(f"مبارك : {r_tool['response'][:300]}")
print(f"Outils : {r_tool['tools_used']}")
print("→ Capture 2 : appel get_weather à prendre dans LangSmith")

print("\n=== Test LangSmith — Mémoire épisodique ===")
THREAD_LS = "langsmith-memory-demo"
chat_with_mbark("أنا محمد، تاجر جاي من كيفة", thread_id=THREAD_LS)
r_mem = chat_with_mbark("شتنصحني في التجارة يا مبارك؟", thread_id=THREAD_LS)
print(f"مبارك : {r_mem['response'][:300]}")
print("→ Capture 3 : vérifier que مبارك mentionne 'محمد' ou 'كيفة'")

---
## Exercice 7 : Pousser sur Hugging Face Hub

In [ ]:
# Décommentez et exécutez après notebook_login()

# from huggingface_hub import notebook_login, HfApi
# from datasets import Dataset
# import pandas as pd
#
# notebook_login()
#
# HF_USERNAME = "votre-username"  # ← changer ici
#
# # Option A : pousser le dataset TP2
# if tp2_data:
#     df = pd.DataFrame(tp2_data)
#     hf_dataset = Dataset.from_pandas(df)
#     hf_dataset.push_to_hub(f"{HF_USERNAME}/mbark-hassaniya-agent-tp3")
#     print(f"Dataset poussé : https://huggingface.co/datasets/{HF_USERNAME}/mbark-hassaniya-agent-tp3")
#
# # Option B : pousser le modèle fine-tuné du TP2
# # from transformers import AutoModelForCausalLM, AutoTokenizer
# # model = AutoModelForCausalLM.from_pretrained("./mbark_model")
# # tokenizer = AutoTokenizer.from_pretrained("./mbark_model")
# # model.push_to_hub(f"{HF_USERNAME}/mbark-hassaniya-tp2-model")
# # tokenizer.push_to_hub(f"{HF_USERNAME}/mbark-hassaniya-tp2-model")

print("Instructions HuggingFace Hub :")
print("  1. notebook_login() dans Colab")
print("  2. Décommenter les lignes ci-dessus")
print("  3. Changer HF_USERNAME par votre nom d'utilisateur")
print("  4. La model card doit décrire : langue, outils, limites, risques d'hallucination")

---
## Exercice 8 : Interface Streamlit

Le fichier `app_tp3.py` contient l'interface complète. Elle affiche :
- La question utilisateur
- La réponse de مبارك
- L'action choisie (outil appelé)
- Les sources utilisées
- Le niveau de confiance
- Un identifiant de trace LangSmith

Pour lancer : `streamlit run app_tp3.py`

In [ ]:
import os, subprocess, time

# Vérification
if os.path.exists("app_tp3.py"):
    print("✅ app_tp3.py présent")
else:
    print("⚠️  app_tp3.py manquant")

# ── Étape 1 : installer streamlit si absent ───────────────────────────────────
import shutil
if not shutil.which("streamlit"):
    print("Installation de streamlit...")
    os.system("pip install -q streamlit")
    print("✅ streamlit installé")

# ── Étape 2 : lancer Streamlit en arrière-plan ───────────────────────────────
proc = subprocess.Popen([
    "streamlit", "run", "app_tp3.py",
    "--server.port", "8501",
    "--server.headless", "true",
    "--server.enableCORS", "false",
    "--server.enableXsrfProtection", "false",
    "--browser.gatherUsageStats", "false"
])
time.sleep(6)
print("✅ Streamlit démarré (port 8501)")

# ── Étape 3 : obtenir l'URL publique — proxy natif Colab ─────────────────────
try:
    from google.colab.output import eval_js
    url = eval_js("google.colab.kernel.proxyPort(8501)")
    print(f"\n✅ Interface disponible ici :\n{url}")
except Exception as e:
    # Fallback : localtunnel (gratuit, sans compte)
    print(f"Proxy Colab non disponible ({e})\nEssai avec localtunnel...")
    os.system("npm install -g localtunnel -q")
    lt = subprocess.Popen(
        ["npx", "localtunnel", "--port", "8501"],
        stdout=subprocess.PIPE, text=True
    )
    time.sleep(3)
    line = lt.stdout.readline().strip()
    print(f"\n✅ Interface disponible ici :\n{line}")
    print("(Si une page 'Click to Continue' apparaît, cliquez dessus)")

---
## Exercice 9 : Démonstration complète et observation des décisions

In [ ]:
print("=" * 65)
print("DÉMONSTRATION COMPLÈTE — Agent مبارك ولد حميدة")
print("=" * 65)

demo_cases = [
    # (question, thread_id, description)
    ("شماسي الطقس في نواكشوط اليوم؟",          "demo-1", "Météo locale"),
    ("  المغرب اليوم في نواكشوط اينت ؟",   "demo-2", "Horaires prières"),
    ("كم يساوي 200 اوقية بالدولار؟",           "demo-3", "Conversion devises"),
    ("عطني مثل حساني عن التجارة والصدق",     "demo-4", "Proverbe (dataset local)"),
    (" سوق العاصمة في نواكشوط منين؟",           "demo-5", "Géolocalisation"),
    ("شنه اشبه وقت لبيع في السوق؟",   "demo-6", "Conseil (mémoire TP2)"),
]

for q, tid, desc in demo_cases:
    print(f"\n[{desc}]")
    print(f"❓ {q}")
    result = chat_with_mbark(q, thread_id=tid)
    formatted = format_source_answer(
        response=result["response"],
        action=result["action"],
        sources=result["sources"],
        confidence=result["confidence"]
    )
    print(f"💬 {formatted['response'][:280]}")
    print(f"   Action    : {formatted['action']}")
    print(f"   Sources   : {formatted['sources']}")
    print(f"   Confiance : {formatted['confidence']}")

---
## Résumé du TP3

| Exercice | Réalisé |
|----------|---------|
| Ex1 — Carte du personnage | ✅ Profil مبارك ولد حميدة injecté dans le prompt système |
| Ex2 — Outils vivants (≥4, ≥1 API externe) | ✅ 6 outils : Open-Meteo, AlAdhan, Frankfurter, proverbes, Nominatim, RAG TP2 |
| Ex3 — Agent `create_react_agent` | ✅ Agent LangGraph avec checkpointer |
| Ex4 — Mémoire épisodique | ✅ Séparation par `thread_id`, démonstration 2 threads |
| Ex5 — Sources HuggingFace | ✅ Dataset TP2 + proverbes + ahmed02mk/amthal-hassaniya (optionnel) |
| Ex6 — LangSmith | ✅ Tracing activé, projet `tp3-mbark-hassaniya-agent` |
| Ex7 — HuggingFace Hub | ✅ Instructions et code fournis (décommenter après login) |
| Ex8 — Interface Streamlit | ✅ `app_tp3.py` avec réponse + action + sources + confiance |

**Lien HuggingFace** : `https://huggingface.co/votre-username/mbark-hassaniya-agent-tp3` *(à compléter après push)*

**Projet LangSmith** : `https://smith.langchain.com` → projet `tp3-mbark-hassaniya-agent`